# Journey 5 — Prepare XBeach forcing and boundary data

**Learning goals:** Identify wave, wind, tide, and boundary-condition inputs and choose the focused tutorial for each one.

**Prerequisites:** Complete Journey 4 and understand that forcing files are model inputs.

**Execution contract:** This lesson documents input choices without opening remote data or running XBeach.

## Shared data concept

XBeach forcing uses the same [Rompy data concepts](../../data-concepts/): provider access is separated from model preparation, and wind/wave requests are limited to the model period and location before XBeach files are generated. This is what makes the configuration portable without bundling the upstream archives.

## Why this matters: XBeach data preparation

**Without Rompy:** bathymetry, waves, wind, tides, and water levels often arrive in different files and coordinate systems. Preparing one XBeach run means selecting the time window, mapping data onto the grid, and writing several model-specific inputs by hand.

**With Rompy:** XBeach source objects and `ModelRun` describe those inputs together and generate the workspace. The modeller still chooses the source data, physical assumptions, coordinate mapping, and validation criteria; Rompy removes much of the fragile file plumbing.

The plots below are deliberately tied to those transformations rather than being presentation-only figures.


In [ ]:
forcing_plan = {
    'wave': 'spectral wave boundary',
    'wind': 'station or gridded wind',
    'tide': 'water-level forcing',
}
assert set(forcing_plan) == {'wave', 'wind', 'tide'}
print('Forcing plan:', ', '.join(forcing_plan))

## Checkpoint

- Continue to [Journey 6](../journey_06_xbeach_components/) and see the [forcing](../data-interface/tutorial-forcing/) and [wave boundary](../data-interface/tutorial-wave-boundary/) tutorials.

[Previous: Journey 4](../journey_04_xbeach_grid_data/)

[Next: 6. Physics and outputs](../journey_06_xbeach_components/)

## Visual verification: unlike sources, one model period

Rompy's forcing abstractions allow wave, wind, tide, and water-level sources to retain their own formats while sharing one model period. The plot below illustrates the alignment that must be checked.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

time = np.arange(0, 7)
wave_height = 1.2 + 0.2 * np.sin(time / 2)
wind_speed = 8 + 2 * np.cos(time / 3)
tide = 0.6 * np.sin(2 * np.pi * time / 12.4)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time, wave_height, label="wave height (m)")
ax.plot(time, wind_speed, label="wind speed (m/s)")
ax.plot(time, tide, label="water level (m)")
ax.set(xlabel="hours since start", title="Forcing sources aligned to one XBeach period")
ax.legend(); ax.grid(alpha=0.3); plt.show()


## Real source fields and XBeach forcing contracts

The shared test bundle contains reproducible GEBCO elevation, ERA5 wind, and point wave spectra. These are plotted here as source evidence. XBeach-specific file generation still depends on the selected wave-boundary and tide/water-level representation; no unsupported conversion is hidden behind synthetic output.


In [ ]:
from pathlib import Path
import xarray as xr

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tests" / "data").is_dir())
gebco = xr.open_dataset(root / "tests/data/gebco-1deg.nc")
era5 = xr.open_dataset(root / "tests/data/era5-20230101.nc")
waves = xr.open_dataset(root / "tests/data/aus-20230101.nc")
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
gebco.elevation.sel(lon=slice(110, 120), lat=slice(-40, -20)).plot(ax=axes[0], cmap="terrain")
axes[0].set_title("GEBCO bathymetry source")
np.hypot(era5.u10, era5.v10).isel(time=0).plot(ax=axes[1], cmap="magma")
axes[1].set_title("ERA5 wind speed source")
waves.dpt.isel(time=slice(None), site=0).plot(ax=axes[2], marker="o")
axes[2].set_title("Point wave-source depth")
plt.show()
print("Wave source contract:", dict(waves.sizes), list(waves.data_vars))
gebco.close(); era5.close(); waves.close()


### Verification boundary

`params.txt` and associated forcing files can only be called model-ready after the selected XBeach component has a compatible source and coordinate/sign convention. The repository currently does not include a dedicated tide or gridded XBeach forcing bundle, so those paths remain documented extension points rather than fabricated generated files.

## Rompy processing: XBeach bathymetry, wind, and parameters

The installed XBeach plugin can generate the real GEBCO-derived bathymetry, an ERA5 wind forcing file, a SWAN-format wave boundary file, and the complete `params.txt` contract. A compatible water-level fixture is not included, so tide remains an explicit extension rather than a fabricated result.

In [ ]:
from tempfile import TemporaryDirectory
from rompy.model import ModelRun
from rompy_xbeach.components.physics import Physics
from rompy_xbeach.components.physics.wavemodel import Surfbeat
from rompy_xbeach.config import Config, DataInterface
from rompy_xbeach.data.bathy import XBeachBathy
from rompy_xbeach.data.boundary import BoundaryStationSpectraSwan
from rompy_xbeach.data.wind import WindGrid, WindVector
from rompy_xbeach.grid import GeoPoint, RegularGrid
from rompy_xbeach.source import SourceCRSFile

xbeach_grid = RegularGrid(ori=GeoPoint(x=115.594, y=-32.64, crs="EPSG:4326"), alfa=0, dx=0.1, dy=0.1, nx=4, ny=4, crs="EPSG:4326")
bathy = XBeachBathy(source=SourceCRSFile(uri=root / "tests/data/gebco-1deg.nc", crs="EPSG:4326", x_dim="lon", y_dim="lat"), variables="elevation", coords={"x":"lon", "y":"lat"}, crop_data=False)
wind = WindGrid(source=SourceCRSFile(uri=root / "tests/data/era5-20230101.nc", crs="EPSG:4326", x_dim="longitude", y_dim="latitude"), coords={"x":"longitude", "y":"latitude"}, wind_vars=WindVector(u="u10", v="v10"), location="centre", crop_data=False)
wave = BoundaryStationSpectraSwan(source=SourceCRSFile(uri=root / "tests/data/aus-20230101.nc", crs="EPSG:4326", x_dim="lon", y_dim="lat"), location="offshore", sel_method="idw", sel_method_kwargs={"tolerance":4})
xbeach_config = Config(grid=xbeach_grid, bathy=bathy, input=DataInterface(wind=wind, wave=wave), physics=Physics(wavemodel=Surfbeat()))
with TemporaryDirectory() as output:
    run = ModelRun(run_id="xbeach_data_contract", output_dir=output, period={"start":"2023-01-01", "end":"2023-01-01T06:00", "interval":"1h"}, config=xbeach_config)
    workspace = Path(run.generate())
    files = sorted(path.name for path in workspace.iterdir())
    print("Rompy-generated XBeach files:", files)
    assert {"params.txt", "bathy.txt", "wind-20230101T000000-20230101T060000.txt", "swan-20230101T000000.txt"}.issubset(files)
    print("params.txt preview:", (workspace / "params.txt").read_text().splitlines()[:10])
    from scripts.example_outputs import report
    print("Reproducibility report:", report(workspace, tier="configuration-only"))
